In [27]:
import os 
import certifi
from dotenv import load_dotenv

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain.tools import tool
import requests

In [28]:
os.environ['SSL_CERT_FILE'] = certifi.where()
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY =os.getenv("WEATHERSTACK_API_KEY")

In [29]:
search_tool = TavilySearchResults(max_results=1)

In [30]:
result = search_tool.invoke('Where is Jagannath  temple?')
result

[{'title': 'Jagannath Temple, Puri - Wikipedia',
  'url': 'https://en.wikipedia.org/wiki/Jagannath_Temple,_Puri',
  'content': 'The Jagannath Temple is a Hindu temple dedicated to Jagannath, a form of Vishnu. It is located in Puri, Odisha, on the eastern coast of India. As per temple records, King Indradyumna of Avanti "Avanti (region)") built the main temple. The present temple complex was rebuilt from the eleventh century onwards on the site of the earlier shrines, excluding the main Jagannath temple, and was begun by Anantavarman Chodaganga, the first ruler of the Eastern Ganga dynasty. Many of the temple rituals are based on Shabari Tantras which are evolved from tribal beliefs respectively. The local legends link the idols with Nilamadhaba deva worshipped by tribala and the daitapatis (servitors) claim to be descendants of the tribes. The temple is one of the 108 Abhimana Kshethram of the Sri Vaishnavite [...] | Jagannath Temple | |\n --- |\n| Shri Jagannath Temple, Puri | |\n| Ja

In [31]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [32]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    api_key = GROQ_API_KEY
)

In [33]:
response = llm.invoke('Where is nalanda?')
response

AIMessage(content='**Nalanda** is an ancient historic site in the Indian subcontinent, best known for the ruins of the world‑famous Nalanda University that flourished from the 5th to the 12th\u202fcentury\u202fCE.\n\n| Detail | Information |\n|--------|--------------|\n| **Country** | India |\n| **State** | Bihar |\n| **District** | Nalanda district (named after the site) |\n| **Nearest towns/cities** | • **Nalanda town** – the village that directly adjoins the ruins (≈\u202f2\u202fkm away) <br>• **Rajgir** – a historic city about 12\u202fkm to the north‑west <br>• **Patna** – the state capital, roughly 90\u202fkm (≈\u202f55\u202fmi) to the north‑west |\n| **Geographic coordinates** | **Latitude:**\u202f25.1400°\u202fN <br>**Longitude:**\u202f85.3800°\u202fE |\n| **Accessibility** | • **By road:** National Highway\u202f20 (formerly NH‑2) connects Patna → Rajgir → Nalanda. <br>• **By rail:** The nearest railway station is **Rajgir (RJGR)**, about 12\u202fkm away; Patna Junction is a maj

In [34]:
tools = [search_tool,get_weather_data]

In [35]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a web research assistant. "
        "Use the search tool for current events. "
        "Base your answers on retrieved sources. "
        "Distinguish verified facts from allegations, "
        "and never invent sources, names, dates, or statistics."
    ),
)

In [36]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Find the latest news about protests in New Delhi."
        }
    ]
})

print(response["messages"][-1].content)

KeyboardInterrupt: 